In [ ]:
import numpy as np
import pandas as pd
import GPy

In [ ]:
# ========================
# 1. Load input data
# ========================

# Low-fidelity CFD-based data (Mach, AoA, Q_flutter)
low_fidelity_data = pd.read_csv("Dataset\\Predictions\\q_flutter_NN_exp_data.csv", sep=',')
X_lf = low_fidelity_data[["Mach", "AoA", "Q_flutter_psf"]].values

# High-fidelity experimental flutter pressure data
experimental_data = pd.read_csv("Dataset\\q_flutter_psf_exp.csv", sep=',')
Y_exp = np.round(experimental_data["q_flutter_psf"].values, decimals=3)
Y = np.expand_dims(Y_exp, axis=1)  # Convert to column vector

# Data points to predict (subset in range: Mach 0.74–0.84, AoA 0–5°)
prediction_data = pd.read_csv("Dataset\\Predictions\\q_flutter_NN_exp_data_074M084_0AoA5.csv", sep=',')
prediction_data = prediction_data[
    (prediction_data['Mach'] >= 0.74) & (prediction_data['Mach'] <= 0.84) &
    (prediction_data['AoA'] >= 0) & (prediction_data['AoA'] <= 5)
]
X_pred = prediction_data[["Mach", "AoA", "Q_flutter_psf"]].values

In [ ]:
# ========================
# 2. Train Gaussian Process model using GPy
# ========================

# Define RBF and Matern kernels
kernel_rbf = GPy.kern.RBF(input_dim=3, variance=1.0, lengthscale=1.0, ARD=True)
kernel_matern = GPy.kern.Matern52(input_dim=3, variance=1.0, lengthscale=1.0, ARD=True)

# Combined kernel: product of RBF and Matern
kernel = kernel_rbf * kernel_matern

# Fit GP model to low-fidelity inputs and experimental outputs
model = GPy.models.GPRegression(X_lf, Y, kernel=kernel)
model.optimize(optimizer='lbfgsb', max_iters=1000)

In [ ]:
# ========================
# 3. Make predictions
# ========================

# Predict Q_flutter at new Mach/AoA points
Y_pred, std_prediction = model.predict(X_pred)

# ========================
# 4. Save results
# ========================

# Extract Mach and AoA values from prediction set
Mach = prediction_data["Mach"].values
AoA = prediction_data["AoA"].values

# Create dataframe with predictions
results_df = pd.DataFrame({
    'Mach': Mach,
    'AoA': AoA,
    'Q_flutter_psf': np.round(Y_pred[:, 0], decimals=3)
})

# Export predicted flutter pressures
results_df.to_csv('q_flutter_data_fusion.csv', sep=',', index=False)
